##E2.1 select, drop, rename — SETUP

In [0]:
%py

df_propiedades=spark.table("bootcamp.silver.propiedades")


df_propiedades.printSchema()

print(f"Total registros bootcamp.silver.propiedades: {df_propiedades.count()}")

In [0]:
%py
from pyspark.sql.functions import col

df_select_1=df_propiedades.select("propiedad_id","precio","moneda").limit(5)

df_select_exp=df_propiedades.select(col("propiedad_id").alias("id"),col("precio").alias("$"),col("moneda").alias("money")).limit(5)

print(f"DF_SELECT_1")
df_select_1.show()
print("\n")
print(f"DF_SELECT_EXP")
df_select_exp.show()  


In [0]:
%py

# eliminar 2 col del df
df_droped=df_select_1.drop("moneda","precio")
print(f"DF_SELECT_1")
df_select_1.show()
print("\n")
print(f"DF_DROPED")
df_droped.show()


In [0]:
%py

df_renamed=df_select_exp.withColumnRenamed("$","price")

df_renamed.show()

##E2.2 filter variado

In [0]:
%py

df_silver_propiedades=spark.table("bootcamp.silver.propiedades")

#igualdad
df_silver_propiedades.filter(col("tipo_operacion")=="venta")

#rango
print(f"BETWEEN\n")
df_silver_propiedades.filter(col("metros_cuadrados_totales").between(20,33)).select("propiedad_id","metros_cuadrados_totales").orderBy("propiedad_id").show(5)

#contain
print(f"CONTAINS\n")
df_silver_propiedades.filter(col("partido").contains("tig")).select("propiedad_id","partido").show(5)

#lista de valores
df_silver_propiedades.filter(col("tipo_operacion").isin("alquiler"))

#negacion
df_silver_propiedades.filter(col("fecha_publicacion")!="2025-09-10")




##E2.3 pipeline con explain

In [0]:
%py

df_silver_props=spark.table("bootcamp.silver.propiedades")

df_filtered=df_silver_props.filter((col("tipo_operacion")=="venta") & (col("moneda")=="USD") & (col("metros_cuadrados_totales")>100)).select("partido","precio","metros_cuadrados_totales","precio_por_m2","ambientes")

df_filtered.explain(True)

##E2.4 Eficiente vs anti-patrón

In [0]:
%py
from pyspark.sql.functions import col
df_silver_props=spark.table("bootcamp.silver.propiedades")

print(f"total_propiedades silver: {df_silver_props.count()}")

df_filtered=df_silver_props.filter((col("tipo_operacion")=="venta") & (col("moneda")=="USD") & (col("metros_cuadrados_totales")>100)).select("partido","precio","metros_cuadrados_totales","precio_por_m2","ambientes")

print(f"total_propiedades silver_filtered: {df_filtered.count()}")

In [0]:
%py
from pyspark.sql.functions import col
df_silver_props=spark.table("bootcamp.silver.propiedades")

# print(f"total_propiedades silver: {df_silver_props.count()}")

df_filtered=df_silver_props.filter((col("tipo_operacion")=="venta") & (col("moneda")=="USD") & (col("metros_cuadrados_totales")>100)).select("partido","precio","metros_cuadrados_totales","precio_por_m2","ambientes")

print(f"total_propiedades silver_filtered: {df_filtered.count()}")

##E2.5 withColumn + when

In [0]:
%python
from pyspark.sql.functions import col,when
df_derivada=(
    df_filtered
    .withColumn("categoria_precio",
                when(col("precio")<80000,"economica")
                .when(col("precio").between(80000,200000),"media")
                .when(col("precio").between(200000,500000),"premium")
                .otherwise("lujo"))
)

df_derivada.limit(20).show()

##E2.6 Pipeline completo

In [0]:
%py
from pyspark.sql.functions import col,when

df_silver_props_2=spark.table("bootcamp.silver.propiedades")


df_filtered=(
    df_silver_props_2
    .filter(col("tipo_operacion")=="alquiler")
    .filter(col("moneda")=="ARS")
    .filter(col("ambientes")>0)
    .select("propiedad_id","partido","precio","ambientes","metros_cuadrados_totales")
    .withColumn("precio_por_ambiente", col("precio")/col("ambientes"))   
    .orderBy(col("precio_por_ambiente").desc())
    .limit(20)
)


#df_filtered.explain(True)

df_filtered.show()


##E2.7 spark.sql() vs DataFrame API

In [0]:
%py

query=spark.sql(f"""
                SELECT
                    propiedad_id,
                    partido,
                    precio,
                    ambientes,
                    metros_cuadrados_totales as metros,
                    precio / ambientes as precio_por_ambiente
                FROM bootcamp.silver.propiedades
                WHERE tipo_operacion = "alquiler" AND moneda = "ARS" AND ambientes > 0
                ORDER BY precio_por_ambiente DESC
                LIMIT 20
                """)

#query.explain(True)

query.show()